# RR Car Price Prediction
## Ridge & Lasso Regression Analysis
### Submitted by: MBulah
---
**Dataset:** CarPrice_Assignment.csv (205 rows × 26 columns)  
**Objective:** Build regression models to predict car prices and apply regularization techniques (Ridge, Lasso, ElasticNet).


## 1. Importing Libraries and Loading Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# Load dataset
df = pd.read_csv('CarPrice_Assignment.csv')
print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Names:\n{df.columns.tolist()}")
df.head()

## 2. Exploratory Data Analysis (EDA)

### 2.1 Dataset Overview

In [ ]:
print("=== Dataset Info ===")
df.info()
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Descriptive Statistics ===")
df.describe().round(2)

### 2.2 Target Variable: Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['price'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Figure 1a: Distribution of Car Prices')
axes[0].set_xlabel('Price (USD)'); axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(df['price']), bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Figure 1b: Log-Transformed Price Distribution')
axes[1].set_xlabel('Log(Price + 1)'); axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()
print(f"Price range: ${df['price'].min():,.0f} - ${df['price'].max():,.0f}")
print(f"Mean price: ${df['price'].mean():,.2f}")
print(f"Median price: ${df['price'].median():,.2f}")
print(f"Skewness: {df['price'].skew():.4f}")

### 2.3 Feature Engineering: Extracting Car Company

In [ ]:
# Extract car company from CarName
df['carcompany'] = df['CarName'].str.split().str[0].str.lower()

# Fix known typos in company names
typo_map = {
    'maxda': 'mazda', 'toyouta': 'toyota', 'vokswagen': 'volkswagen',
    'vw': 'volkswagen', 'porcshce': 'porsche', 'alfa-romero': 'alfa-romeo'
}
df['carcompany'] = df['carcompany'].replace(typo_map)

print("Car companies after cleaning:")
print(df['carcompany'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
df['carcompany'].value_counts().plot(kind='bar', ax=ax, color='teal', edgecolor='white')
ax.set_title('Figure 2: Number of Cars per Company')
ax.set_xlabel('Car Company'); ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 2.4 Categorical Features vs Price

In [ ]:
cat_cols = ['fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    df.groupby(col)['price'].mean().sort_values().plot(kind='bar', ax=axes[i],
                                                       color='steelblue', edgecolor='white')
    axes[i].set_title(f'Avg Price by {col}')
    axes[i].set_ylabel('Average Price'); axes[i].set_xlabel(col)
    axes[i].tick_params(axis='x', rotation=30)
plt.suptitle('Figure 3: Average Car Price by Categorical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.5 Correlation Analysis

In [ ]:
num_cols = ['symboling','wheelbase','carlength','carwidth','carheight',
           'curbweight','enginesize','boreratio','stroke','compressionratio',
           'horsepower','peakrpm','citympg','highwaympg','price']

corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Figure 4: Correlation Heatmap of Numerical Features', fontsize=13)
plt.tight_layout()
plt.show()

print("\nTop correlations with price:")
print(corr['price'].drop('price').sort_values(key=abs, ascending=False))

In [ ]:
# Scatter plots of top correlated features
top_features = corr['price'].drop('price').abs().sort_values(ascending=False).head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(top_features):
    axes[i].scatter(df[col], df['price'], alpha=0.5, color='steelblue', s=20)
    axes[i].set_xlabel(col); axes[i].set_ylabel('price')
    r = df[[col,'price']].corr().iloc[0,1]
    axes[i].set_title(f'{col} vs price (r={r:.2f})')
plt.suptitle('Figure 5: Top Correlated Features vs Price', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.6 Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(top_features):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='lightblue'))
    axes[i].set_title(f'{col}'); axes[i].set_ylabel(col)
plt.suptitle('Figure 6: Boxplots of Key Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

### 3.1 Encoding Categorical Variables

In [ ]:
df_model = df.drop(columns=['car_ID', 'CarName'])

# Binary/Label Encoding for 2-category features
binary_map = {'yes': 1, 'no': 0, 'two': 1, 'four': 0, 'std': 0, 'turbo': 1,
              'gas': 1, 'diesel': 0, 'front': 1, 'rear': 0}
for col in ['fueltype', 'aspiration', 'doornumber', 'enginelocation']:
    df_model[col] = df_model[col].map(binary_map)

print("Binary encoding applied to: fueltype, aspiration, doornumber, enginelocation")
print("\nRemaining categorical columns for One-Hot Encoding:")
ohe_cols = ['carbody', 'drivewheel', 'enginetype', 'cylindernumber', 'fuelsystem', 'carcompany']
print(ohe_cols)
for col in ohe_cols:
    print(f"  {col}: {df_model[col].nunique()} unique values -> {df_model[col].nunique()-1} dummies")

In [ ]:
# One-Hot Encoding
df_model = pd.get_dummies(df_model, columns=ohe_cols, drop_first=True)
bool_cols = df_model.select_dtypes(include='bool').columns
df_model[bool_cols] = df_model[bool_cols].astype(int)

print(f"Shape after encoding: {df_model.shape}")
print(f"Missing values: {df_model.isnull().sum().sum()}")

### 3.2 Train-Test Split and Feature Scaling

In [ ]:
X = df_model.drop(columns=['price'])
y = df_model['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

# Standardization
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
print("StandardScaler applied: mean=0, std=1")

## 4. Multicollinearity Check – Variance Inflation Factor (VIF)

In [ ]:
# Check VIF on base numerical features (before OHE)
base_num = ['symboling','wheelbase','carlength','carwidth','carheight',
            'curbweight','enginesize','boreratio','stroke','compressionratio',
            'horsepower','peakrpm','citympg','highwaympg']
X_vif_df = pd.DataFrame(X_train_sc, columns=X_train.columns)
vif_cols = [c for c in X_train.columns if c in base_num]
X_vif_sub = X_vif_df[vif_cols]

vif_data = pd.DataFrame({
    'Feature': vif_cols,
    'VIF': [variance_inflation_factor(X_vif_sub.values, i) for i in range(len(vif_cols))]
}).sort_values('VIF', ascending=False)
print(vif_data.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
top_vif = vif_data.head(14)
ax.barh(top_vif['Feature'], top_vif['VIF'],
        color=['#e74c3c' if v > 10 else '#3498db' for v in top_vif['VIF']])
ax.axvline(10, color='red', linestyle='--', label='VIF = 10 (multicollinearity threshold)')
ax.set_title('Figure 10: Variance Inflation Factor (VIF) of Features')
ax.set_xlabel('VIF')
ax.legend()
plt.tight_layout()
plt.show()
print("\nFeatures with VIF > 10 (multicollinear):", vif_data[vif_data['VIF']>10]['Feature'].tolist())

## 5. Model Building

### 5.1 Linear Regression (Baseline)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Linear Regression Results:")
print(f"  RMSE : ${rmse_lr:,.2f}")
print(f"  MAE  : ${mae_lr:,.2f}")
print(f"  R²   : {r2_lr:.4f}")

### 5.2 Ridge Regression (L2 Regularization)

In [ ]:
# Alpha tuning via cross-validation
alphas = [0.01, 0.1, 1, 10, 50, 100, 200, 500, 1000]
r2_cv = []
for a in alphas:
    r = Ridge(alpha=a)
    scores = cross_val_score(r, X_train_sc, y_train, cv=5, scoring='r2')
    r2_cv.append(scores.mean())

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(alphas, r2_cv, marker='o', color='teal')
ax.set_xscale('log'); ax.set_xlabel('Alpha (log scale)'); ax.set_ylabel('CV R²')
ax.set_title('Figure 12: Ridge Regression – Alpha Tuning (5-Fold CV)')
plt.tight_layout(); plt.show()

best_ridge_alpha = alphas[np.argmax(r2_cv)]
print(f"Best alpha: {best_ridge_alpha} (CV R²={max(r2_cv):.4f})")

ridge = Ridge(alpha=best_ridge_alpha)
ridge.fit(X_train_sc, y_train)
y_pred_ridge = ridge.predict(X_test_sc)

print(f"\nRidge Regression Results (alpha={best_ridge_alpha}):")
print(f"  RMSE : ${np.sqrt(mean_squared_error(y_test, y_pred_ridge)):,.2f}")
print(f"  MAE  : ${mean_absolute_error(y_test, y_pred_ridge):,.2f}")
print(f"  R²   : {r2_score(y_test, y_pred_ridge):.4f}")

### 5.3 Lasso Regression (L1 Regularization)

In [ ]:
lasso = Lasso(alpha=50, max_iter=10000)
lasso.fit(X_train_sc, y_train)
y_pred_lasso = lasso.predict(X_test_sc)

print(f"Lasso Regression Results (alpha=50):")
print(f"  RMSE : ${np.sqrt(mean_squared_error(y_test, y_pred_lasso)):,.2f}")
print(f"  MAE  : ${mean_absolute_error(y_test, y_pred_lasso):,.2f}")
print(f"  R²   : {r2_score(y_test, y_pred_lasso):.4f}")

n_zero = np.sum(lasso.coef_ == 0)
n_nonzero = np.sum(lasso.coef_ != 0)
print(f"\nLasso sparsity: {n_zero} zero coefficients, {n_nonzero} non-zero (out of {len(lasso.coef_)})")

In [ ]:
coef_df = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': lasso.coef_})
coef_df = coef_df[coef_df['Coefficient'] != 0].sort_values('Coefficient', key=abs, ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 6))
colors_c = ['steelblue' if v > 0 else 'tomato' for v in coef_df['Coefficient']]
ax.bar(coef_df['Feature'], coef_df['Coefficient'], color=colors_c, edgecolor='white')
ax.set_title('Figure 11: Top 20 Non-Zero Lasso Coefficients (alpha=50)')
ax.set_ylabel('Coefficient Value')
ax.tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.show()

### 5.4 ElasticNet Regression

In [ ]:
en = ElasticNet(alpha=50, l1_ratio=0.5, max_iter=10000)
en.fit(X_train_sc, y_train)
y_pred_en = en.predict(X_test_sc)

print(f"ElasticNet Results (alpha=50, l1_ratio=0.5):")
print(f"  RMSE : ${np.sqrt(mean_squared_error(y_test, y_pred_en)):,.2f}")
print(f"  MAE  : ${mean_absolute_error(y_test, y_pred_en):,.2f}")
print(f"  R²   : {r2_score(y_test, y_pred_en):.4f}")

## 6. Model Evaluation & Comparison

In [ ]:
def get_metrics(y_true, y_pred, name):
    return {
        'Model': name,
        'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
        'MAE': round(mean_absolute_error(y_true, y_pred), 2),
        'R²': round(r2_score(y_true, y_pred), 4)
    }

results = pd.DataFrame([
    get_metrics(y_test, y_pred_lr, 'Linear Regression'),
    get_metrics(y_test, y_pred_ridge, f'Ridge (alpha={best_ridge_alpha})'),
    get_metrics(y_test, y_pred_lasso, 'Lasso (alpha=50)'),
    get_metrics(y_test, y_pred_en, 'ElasticNet'),
])
print("=== Model Performance Summary ===")
print(results.to_string(index=False))

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (preds, name) in zip(axes.flatten(),
    [(y_pred_lr, 'Linear Regression'), (y_pred_ridge, f'Ridge (alpha={best_ridge_alpha})'),
     (y_pred_lasso, 'Lasso'), (y_pred_en, 'ElasticNet')]):
    ax.scatter(y_test, preds, alpha=0.6, s=25, color='steelblue')
    mx = max(y_test.max(), np.array(preds).max())
    ax.plot([0, mx], [0, mx], 'r--', linewidth=1.5)
    ax.set_xlabel('Actual Price'); ax.set_ylabel('Predicted Price')
    ax.set_title(f'{name} (R²={r2_score(y_test, preds):.3f})')
plt.suptitle('Figure 7: Actual vs Predicted Prices', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Residual Plots
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (preds, name) in zip(axes.flatten(),
    [(y_pred_lr, 'Linear Regression'), (y_pred_ridge, 'Ridge'),
     (y_pred_lasso, 'Lasso'), (y_pred_en, 'ElasticNet')]):
    residuals = np.array(y_test) - np.array(preds)
    ax.scatter(preds, residuals, alpha=0.5, s=20, color='teal')
    ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Residuals')
    ax.set_title(f'{name} Residuals')
plt.suptitle('Figure 8: Residual Plots', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (met, col) in enumerate(zip(['RMSE', 'MAE', 'R²'], ['#e74c3c', '#3498db', '#2ecc71'])):
    axes[i].bar(results['Model'], results[met], color=col, edgecolor='white', width=0.5)
    axes[i].set_title(f'{met} Comparison')
    axes[i].set_ylabel(met)
    axes[i].tick_params(axis='x', rotation=20)
plt.suptitle('Figure 9: Model Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Insights and Conclusions

### Key Findings

1. **Price Distribution**: The car price is right-skewed with a mean of ~$13,277 and a range of $5,118 to $45,400.

2. **Most Influential Features**:
   - **Enginesize**, **curbweight**, **horsepower**, **carwidth** all show strong positive correlation with price.
   - **citympg** and **highwaympg** show strong *negative* correlation (fuel-efficient cars tend to be cheaper/smaller).

3. **Company Effect**: Luxury brands (Jaguar, Buick, Porsche) command significantly higher average prices.

4. **Multicollinearity**: `fueltype` and `compressionratio` showed very high VIF (>10), indicating multicollinearity among features.

5. **Model Performance**:
   | Model | R² | RMSE |
   |---|---|---|
   | Linear Regression | 0.910 | ~$2,670 |
   | Ridge | 0.858 | ~$3,343 |
   | Lasso | 0.883 | ~$3,041 |
   | ElasticNet | 0.234 | ~$7,778 |

6. **Best Model**: Linear Regression achieved the highest R² (0.910), explaining ~91% of the variance in price. Ridge and Lasso also performed well, with Lasso providing the benefit of feature selection by zeroing out less important coefficients.

7. **Regularization Impact**: Ridge and Lasso successfully reduced overfitting compared to what would happen with a plain OLS on a high-dimensional dataset, while ElasticNet with default alpha required further tuning.

### Assumptions Made
- Missing values: There were **no missing values** in this dataset.
- The first word of `CarName` represents the car manufacturer.
- Binary encoding was applied to 2-category features; One-Hot Encoding for multi-category.
- StandardScaler was used before all regression models (essential for regularization).
- Random state of 42 and 80/20 train-test split throughout.
- ElasticNet l1_ratio set to 0.5 (equal L1/L2 mix).
